In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.artifacts_store import (
    apply_bot_cache,
    save_game_cache,
    train_or_load_bot_detector,
    try_load_game_cache,
)

N_PREVIEW = 5


def save_human_or_bot_cache(game, split, features_df, traces, session_ids, bot_types=None, user_ids=None):
    save_game_cache(
        game=game,
        split=split,
        features_df=features_df,
        traces=traces,
        session_ids=session_ids,
        bot_types=bot_types,
        user_ids=user_ids,
    )


## Load LoL dataset

In [ ]:
from src.data import load_lol_match_windows
from src.config import LOL_MATCH_WINDOW_START_MIN, LOL_MATCH_WINDOW_END_MIN

# first match-aligned window (game clock 10–13 min)
lol_records, lol_load_summary = load_lol_match_windows()
print("LoL match-window summary (smoke):", lol_load_summary)
if lol_records:
    sample_lol = lol_records[0]["mouse"]
    print(f"gameId={lol_records[0]['gameId']}")
    print(
        f"Events: {len(sample_lol)}, duration: {sample_lol['time'].iloc[-1] / 60000:.2f} min "
        f"(target {LOL_MATCH_WINDOW_END_MIN - LOL_MATCH_WINDOW_START_MIN} min @ "
        f"{LOL_MATCH_WINDOW_START_MIN}-{LOL_MATCH_WINDOW_END_MIN})"
    )
    print(sample_lol.head())
else:
    sample_lol = None
    print("No window found for first keylogger (short match or sparse 10–13).")


## LoL extract features

In [ ]:
from src.features import extract_features
from src.config import (
    LOL_MATCH_WINDOW_START_MIN,
    LOL_MATCH_WINDOW_END_MIN,
)

lol_cache = try_load_game_cache("lol", "human")
lol_mouse_by_game = {}
if lol_cache is not None:
    lol_games_df = lol_cache["features"]
    lol_human_traces = lol_cache["traces"]
    for sid, trace in zip(lol_cache["session_ids"], lol_human_traces):
        lol_mouse_by_game[sid] = trace
    print(f"Loaded LoL human cache: features={len(lol_games_df)} traces={len(lol_human_traces)}")
else:
    lol_rows = []
    lol_human_traces = []
    for rec in lol_records:
        lol_mouse = rec["mouse"]
        feats = extract_features(lol_mouse)
        if feats is None:
            continue
        lol_meta = {
            "userId": rec["userId"],
            "gameId": rec["gameId"],
            "source_file": rec["source_file"],
            "session_date": rec["session_date"],
            "match_id": rec["match_id"],
            "participant": rec["participant"],
        }
        feats.update({
            **lol_meta,
            "is_bot": 0,
            "bot_type": "human",
        })
        lol_rows.append(feats)
        lol_human_traces.append(lol_mouse)
        lol_mouse_by_game[rec["gameId"]] = lol_mouse

    lol_games_df = pd.DataFrame(lol_rows)
    save_human_or_bot_cache(
        game="lol",
        split="human",
        features_df=lol_games_df,
        traces=lol_human_traces,
        session_ids=lol_games_df["gameId"].astype(str).tolist(),
        bot_types=["human"] * len(lol_human_traces),
        user_ids=lol_games_df["userId"].astype(str).tolist(),
    )
    print(f"Saved LoL human cache: features={len(lol_games_df)} traces={len(lol_human_traces)}")
print(
    f"Loaded {len(lol_games_df)} LoL human windows "
    f"(match-aligned {LOL_MATCH_WINDOW_START_MIN}–{LOL_MATCH_WINDOW_END_MIN} min)"
)
print(lol_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
re_cache = try_load_game_cache("re", "human")
if re_cache is not None:
    print(f"Red Eclipse — median n_events: {re_cache['features']['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")
print(f"LoL users: {lol_games_df['userId'].nunique()}, matches: {lol_games_df['match_id'].nunique()}")


## LoL trajectory preview

In [ ]:
from src.config import LOL_MATCH_WINDOW_START_MIN, LOL_MATCH_WINDOW_END_MIN
from src.plotting import plot_trajectory

preview_lol = lol_human_traces[0].copy()
print(
    f"Preview: {lol_games_df.iloc[0]['gameId']} | "
    f"{len(preview_lol)} events | {preview_lol['time'].iloc[-1]/60000:.2f} min"
)
plot_trajectory(
    preview_lol,
    title=f"LoL match window {LOL_MATCH_WINDOW_START_MIN}-{LOL_MATCH_WINDOW_END_MIN} min",
)


## LoL Stitch bot generation

In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED

if not lol_human_traces:
    raise RuntimeError('lol_human_traces empty — run LoL match-window feature cell first')

lol_bot_rng = np.random.default_rng(RNG_SEED + 1)
lol_segment_pool = []
for mouse in lol_human_traces:
    lol_segment_pool.extend(build_segments(mouse, rng=lol_bot_rng))

lol_motion = collect_human_motion_samples(lol_human_traces, rng=lol_bot_rng)
lol_target_ms = median_trace_duration_ms(lol_human_traces)
print(
    f"LoL dt n={len(lol_motion['dt_samples'])} sessions={len(lol_motion['dt_by_session'])} median={np.median(lol_motion['dt_samples']):.2f}ms | "
    f"step median={np.median(lol_motion['step_samples']):.3f} | "
    f"target={lol_target_ms/1000:.1f}s"
)

lol_bot_cache = try_load_game_cache('lol', 'bot')
if lol_bot_cache is not None:
    apply_bot_cache('lol', lol_bot_cache, globals(), n_preview=N_PREVIEW)
    LOL_BOTS_READY = True
    print(f"Loaded LoL bot cache: features={len(lol_bot_cache['features'])} traces={len(lol_bot_cache['traces'])}")
else:
    LOL_BOTS_READY = False
    lol_stitch_rows = []
    lol_stitch_traces = []
    sample_lol_stitch_trajectories = []
    for i in range(len(lol_games_df)):
        bot_mouse = stitch_bot_game(
            lol_segment_pool,
            dt_samples=lol_motion['dt_samples'],
            dt_by_session=lol_motion['dt_by_session'],
            target_duration_ms=lol_target_ms,
            rng=lol_bot_rng,
        )
        lol_stitch_traces.append(bot_mouse)
        if len(sample_lol_stitch_trajectories) < N_PREVIEW:
            sample_lol_stitch_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -1, 'gameId': f'lol_stitch_{i}', 'is_bot': 1, 'bot_type': 'stitch'})
        lol_stitch_rows.append(feats)
    lol_bots_stitch_df = pd.DataFrame(lol_stitch_rows)
    print(f"LoL Stitch bots: {len(lol_bots_stitch_df)} (target {lol_target_ms/1000:.1f}s each)")


## LoL Stitch bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_stitch_trajectories):
    print(f"LoL Stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"LoL Stitch_{i}")

## LoL Scripted bot generation

In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_generator_params,
    smooth_params_for_print,
)
from src.features import extract_features

lol_median_events = int(lol_games_df['n_events'].median())
lol_smooth_params = estimate_smooth_params(lol_games_df, **lol_motion)
print(f"LoL Scripted params: {smooth_params_for_print(lol_smooth_params)}")

if "re_smooth_params" not in globals():
    _re_cache = globals().get("re_cache") or try_load_game_cache("re", "human")
    if _re_cache is not None:
        _re_motion = collect_human_motion_samples(
            _re_cache["traces"], rng=np.random.default_rng(RNG_SEED)
        )
        re_smooth_params = estimate_smooth_params(_re_cache["features"], **_re_motion)

if globals().get("re_smooth_params") is not None:
    print(
        f"(Red Eclipse Scripted params for comparison: "
        f"{smooth_params_for_print(re_smooth_params)})"
    )

if not LOL_BOTS_READY:
    lol_smooth_gen = smooth_generator_params(lol_smooth_params)
    lol_smooth_rows = []
    lol_smooth_traces = []
    sample_lol_smooth_trajectories = []
    for i in range(len(lol_games_df)):
        bot_mouse = generate_smooth_bot_game(
            n_events=max(lol_median_events * 3, 1),
            seed=RNG_SEED + 120 + i,
            target_duration_ms=lol_target_ms,
            **lol_smooth_gen,
        )
        lol_smooth_traces.append(bot_mouse)
        if len(sample_lol_smooth_trajectories) < N_PREVIEW:
            sample_lol_smooth_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -2, 'gameId': f'lol_smooth_{i}', 'is_bot': 1, 'bot_type': 'smooth'})
        lol_smooth_rows.append(feats)
    lol_bots_smooth_df = pd.DataFrame(lol_smooth_rows)
    print(f"LoL Scripted bots: {len(lol_bots_smooth_df)} (n_events={lol_median_events})")
else:
    print('LoL Scripted bots loaded from cache')


## LoL Scripted bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_smooth_trajectories):
    plot_trajectory(df, title=f"LoL Scripted_{i}")
    print(f"LoL Scripted_{i}, events={len(df)}")


## LoL Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features

lol_median_events = int(lol_games_df['n_events'].median())
lol_bezier_params = estimate_bezier_params(lol_games_df, **lol_motion)
print(f"LoL Bézier params: {bezier_params_for_print(lol_bezier_params)}")

if "re_bezier_params" not in globals():
    _re_cache = globals().get("re_cache") or try_load_game_cache("re", "human")
    if _re_cache is not None:
        _re_motion = collect_human_motion_samples(
            _re_cache["traces"], rng=np.random.default_rng(RNG_SEED)
        )
        re_bezier_params = estimate_bezier_params(_re_cache["features"], **_re_motion)

if globals().get("re_bezier_params") is not None:
    print(
        f"(Red Eclipse Bézier params for comparison: "
        f"{bezier_params_for_print(re_bezier_params)})"
    )

if not LOL_BOTS_READY:
    lol_bezier_rows = []
    lol_bezier_traces = []
    sample_lol_bezier_trajectories = []
    for i in range(len(lol_games_df)):
        bot_mouse = generate_bezier_bot_game(
            n_events=max(lol_median_events * 3, 1),
            seed=RNG_SEED + 150 + i,
            target_duration_ms=lol_target_ms,
            **lol_bezier_params,
        )
        lol_bezier_traces.append(bot_mouse)
        if len(sample_lol_bezier_trajectories) < N_PREVIEW:
            sample_lol_bezier_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -3, 'gameId': f'lol_bezier_{i}', 'is_bot': 1, 'bot_type': 'bezier'})
        lol_bezier_rows.append(feats)
    lol_bots_bezier_df = pd.DataFrame(lol_bezier_rows)
    print(f"LoL Bézier bots: {len(lol_bots_bezier_df)} (n_events={lol_median_events})")
else:
    print('LoL Bézier bots loaded from cache')


## LoL Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_bezier_trajectories):
    plot_trajectory(df, title=f"LoL Bézier_{i}")
    print(f"LoL Bézier_{i}, events={len(df)}")



## LoL VAE bot — train once / load weights

In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_LOL_WEIGHTS

# False → load artifacts/vae_lol_v4.pt if present; else trains once and saves.
LOL_VAE_FORCE_RETRAIN = False
LOL_VAE_WEIGHTS_PATH = DEFAULT_LOL_WEIGHTS

lol_step_median = float(np.median(lol_motion["step_samples"]))
lol_vae_bundle = ensure_vae_bundle(
    lol_human_traces,
    lol_step_median,
    path=LOL_VAE_WEIGHTS_PATH,
    force_retrain=LOL_VAE_FORCE_RETRAIN,
    seed=RNG_SEED + 1,
)
print(
    f"LoL VAE ready | path={Path(LOL_VAE_WEIGHTS_PATH)} | "
    f"seg_len={lol_vae_bundle['seg_len']} z={lol_vae_bundle['z_dim']} "
    f"norm={lol_vae_bundle.get('norm')} axis_scale={lol_vae_bundle.get('axis_scale')} "
    f"trained_segments={lol_vae_bundle.get('n_segments')}"
)


## LoL VAE bot generation


In [ ]:
from src.vae_bot import generate_vae_bot_games
from src.features import extract_features
from src.config import RNG_SEED, VAE_POOL_SEGMENTS

if not LOL_BOTS_READY:
    lol_vae_rng = np.random.default_rng(RNG_SEED + 4)
    lol_vae_traces = generate_vae_bot_games(
        lol_vae_bundle,
        n_games=len(lol_games_df),
        dt_samples=lol_motion['dt_samples'],
        dt_by_session=lol_motion['dt_by_session'],
        target_duration_ms=lol_target_ms,
        n_pool_segments=VAE_POOL_SEGMENTS,
        rng=lol_vae_rng,
    )
    sample_lol_vae_trajectories = [m.copy() for m in lol_vae_traces[:N_PREVIEW]]
    lol_vae_rows = []
    for i, bot_mouse in enumerate(lol_vae_traces):
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({'userId': -4, 'gameId': f'lol_vae_{i}', 'is_bot': 1, 'bot_type': 'vae'})
        lol_vae_rows.append(feats)
    lol_bots_vae_df = pd.DataFrame(lol_vae_rows)
    print(f"LoL VAE bots: {len(lol_bots_vae_df)} (pool={VAE_POOL_SEGMENTS} segs/game)")
    print(lol_bots_vae_df.head())
else:
    print('LoL VAE bots loaded from cache')


In [ ]:
if not globals().get('LOL_BOTS_READY'):
    lol_bot_features = pd.concat(
        [lol_bots_stitch_df, lol_bots_smooth_df, lol_bots_bezier_df, lol_bots_vae_df], ignore_index=True
    )
    lol_bot_traces = lol_stitch_traces + lol_smooth_traces + lol_bezier_traces + lol_vae_traces
    save_human_or_bot_cache(
        game='lol',
        split='bot',
        features_df=lol_bot_features,
        traces=lol_bot_traces,
        session_ids=lol_bot_features['gameId'].astype(str).tolist(),
        bot_types=lol_bot_features['bot_type'].astype(str).tolist(),
        user_ids=lol_bot_features['userId'].astype(str).tolist(),
    )
    LOL_BOTS_READY = True
    print(f"Saved LoL bot cache: features={len(lol_bot_features)} traces={len(lol_bot_traces)}")


## LoL VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_vae_trajectories):
    plot_trajectory(df, title=f"LoL VAE_{i}")
    print(f"\nlol_vae_{i}, events={len(df)}")


## LoL in-domain (GroupKFold + split-player + windows)

In [ ]:
from src.evaluation import evaluate_group_kfold_windows
from src.features import feature_cols
from src.config import RNG_SEED

def _lol_traces_for(feat_df):
    return [lol_mouse_by_game[gid] for gid in feat_df["gameId"]]

lol_cross_result = evaluate_group_kfold_windows(
    human_df=lol_games_df,
    groups=lol_games_df["userId"].to_numpy(),
    traces_for_df=_lol_traces_for,
    feature_cols=feature_cols,
    rng_seed_base=RNG_SEED + 1,
    vae_bundle=lol_vae_bundle,
    name="LoL",
)
print("\nLoL fold table (window counts):")
print(
    lol_cross_result["fold_df"][
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nLoL window-level summary:")
print(lol_cross_result["summary_df"].to_string(index=False))
if lol_cross_result["session_summary_df"] is not None:
    print("\nLoL session-mean-of-windows summary:")
    print(lol_cross_result["session_summary_df"].to_string(index=False))
